In [1]:
# Participant-level bootstrap confidence interval
# 목적:
# - 최종 후보들의 test 성능 불확실성을 participant 단위 bootstrap으로 추정한다.
# - row 단위가 아니라 participant 단위로 재표본추출한다.
# - 같은 participant 내부 row 상관을 어느 정도 보존한다.

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.metrics import (
    balanced_accuracy_score,
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
)

PROJECT_ROOT = Path(r"c:\workSpace\DeepLearnin_sleep")

PHASE_1A_OUTPUT_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "deep_learning_subset_experiments"
    / "phase_1a_outputs"
)

PHASE_2A_OUTPUT_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "deep_learning_previous_day"
    / "experiments"
    / "phase_2a_outputs"
)

PHASE_1A_PREDICTIONS_PATH = PHASE_1A_OUTPUT_ROOT / "phase_1a_model_predictions.csv"
PHASE_2A_PREDICTIONS_PATH = PHASE_2A_OUTPUT_ROOT / "phase_2a_previous_day_model_predictions.csv"

PHASE_1A_THRESHOLD_PATH = PHASE_1A_OUTPUT_ROOT / "phase_1a_threshold_tuned_metrics.csv"
PHASE_2A_THRESHOLD_PATH = PHASE_2A_OUTPUT_ROOT / "phase_2a_previous_day_threshold_tuned_metrics.csv"

phase1_predictions = pd.read_csv(PHASE_1A_PREDICTIONS_PATH, encoding="utf-8-sig")
phase2_predictions = pd.read_csv(PHASE_2A_PREDICTIONS_PATH, encoding="utf-8-sig")

phase1_threshold = pd.read_csv(PHASE_1A_THRESHOLD_PATH, encoding="utf-8-sig")
phase2_threshold = pd.read_csv(PHASE_2A_THRESHOLD_PATH, encoding="utf-8-sig")

phase1_predictions["feature_timing"] = "same_date"
phase2_predictions["feature_timing"] = "previous_day"

all_predictions = pd.concat(
    [phase1_predictions, phase2_predictions],
    ignore_index=True,
)

all_thresholds = pd.concat(
    [
        phase1_threshold.assign(feature_timing="same_date"),
        phase2_threshold.assign(feature_timing="previous_day"),
    ],
    ignore_index=True,
)

threshold_lookup = (
    all_thresholds[all_thresholds["split"] == "validation"]
    [["feature_timing", "experiment_id", "selected_threshold_from_validation"]]
    .drop_duplicates()
)

BOOTSTRAP_CANDIDATES = [
    {
        "feature_timing": "same_date",
        "experiment_id": "phase1a_003",
        "label": "best_same_date_daytime_only_mlp_w7",
    },
    {
        "feature_timing": "same_date",
        "experiment_id": "phase1a_002",
        "label": "same_date_daytime_only_gru_w7",
    },
    {
        "feature_timing": "same_date",
        "experiment_id": "phase1a_007",
        "label": "same_date_daytime_only_mlp_w14",
    },
    {
        "feature_timing": "previous_day",
        "experiment_id": "phase2a_006",
        "label": "best_previous_day_daytime_only_bilstm_w14",
    },
]

N_BOOTSTRAP = 1000
RANDOM_SEED = 42

rng = np.random.default_rng(RANDOM_SEED)


def compute_binary_metrics(y_true, probability, threshold):
    y_pred = (probability >= threshold).astype(int)

    result = {
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
    }

    if len(np.unique(y_true)) == 2:
        result["roc_auc"] = roc_auc_score(y_true, probability)
        result["average_precision"] = average_precision_score(y_true, probability)
    else:
        result["roc_auc"] = np.nan
        result["average_precision"] = np.nan

    return result


bootstrap_summary_rows = []
bootstrap_detail_rows = []

for candidate in BOOTSTRAP_CANDIDATES:
    feature_timing = candidate["feature_timing"]
    experiment_id = candidate["experiment_id"]
    label = candidate["label"]

    threshold_row = threshold_lookup[
        (threshold_lookup["feature_timing"] == feature_timing)
        & (threshold_lookup["experiment_id"] == experiment_id)
    ]

    if len(threshold_row) != 1:
        raise ValueError(f"threshold lookup 실패: {feature_timing}, {experiment_id}")

    threshold = float(threshold_row["selected_threshold_from_validation"].iloc[0])

    pred = all_predictions[
        (all_predictions["feature_timing"] == feature_timing)
        & (all_predictions["experiment_id"] == experiment_id)
        & (all_predictions["split"] == "test")
    ].copy()

    participants = pred["participant_object_id"].unique()

    original_metrics = compute_binary_metrics(
        pred["y_true"].to_numpy(),
        pred["probability"].to_numpy(),
        threshold,
    )

    bootstrap_metrics = []

    for bootstrap_idx in range(N_BOOTSTRAP):
        sampled_participants = rng.choice(
            participants,
            size=len(participants),
            replace=True,
        )

        sampled_frames = []

        for sampled_participant in sampled_participants:
            sampled_frames.append(
                pred[pred["participant_object_id"] == sampled_participant]
            )

        sampled = pd.concat(sampled_frames, ignore_index=True)

        metrics = compute_binary_metrics(
            sampled["y_true"].to_numpy(),
            sampled["probability"].to_numpy(),
            threshold,
        )

        metrics.update({
            "bootstrap_idx": bootstrap_idx,
            "feature_timing": feature_timing,
            "experiment_id": experiment_id,
            "label": label,
        })

        bootstrap_metrics.append(metrics)

    bootstrap_df = pd.DataFrame(bootstrap_metrics)

    for metric_name in [
        "balanced_accuracy",
        "roc_auc",
        "average_precision",
        "f1",
        "precision",
        "recall",
    ]:
        values = bootstrap_df[metric_name].dropna()

        bootstrap_summary_rows.append({
            "feature_timing": feature_timing,
            "experiment_id": experiment_id,
            "label": label,
            "threshold": threshold,
            "participants": len(participants),
            "test_rows": len(pred),
            "metric": metric_name,
            "original_value": original_metrics[metric_name],
            "bootstrap_mean": values.mean(),
            "ci_lower_2_5": values.quantile(0.025),
            "ci_upper_97_5": values.quantile(0.975),
        })

    bootstrap_detail_rows.append(bootstrap_df)

bootstrap_summary = pd.DataFrame(bootstrap_summary_rows)
bootstrap_details = pd.concat(bootstrap_detail_rows, ignore_index=True)

display(bootstrap_summary)

bootstrap_summary_path = PHASE_2A_OUTPUT_ROOT / "selected_candidate_participant_bootstrap_summary.csv"
bootstrap_details_path = PHASE_2A_OUTPUT_ROOT / "selected_candidate_participant_bootstrap_details.csv"

bootstrap_summary.to_csv(bootstrap_summary_path, index=False, encoding="utf-8-sig")
bootstrap_details.to_csv(bootstrap_details_path, index=False, encoding="utf-8-sig")

print("saved:", bootstrap_summary_path)
print("saved:", bootstrap_details_path)

,feature_timing,experiment_id,label,threshold,participants,test_rows,metric,original_value,bootstrap_mean,ci_lower_2_5,ci_upper_97_5
0,same_date,phase1a_003,best_same_date_daytime_only_mlp_w7,0.42,9,314,balanced_accuracy,0.843989,0.848589,0.804219,0.892402
1,same_date,phase1a_003,best_same_date_daytime_only_mlp_w7,0.42,9,314,roc_auc,0.902264,0.906428,0.848252,0.946265
2,same_date,phase1a_003,best_same_date_daytime_only_mlp_w7,0.42,9,314,average_precision,0.840232,0.830578,0.584970,0.937074
3,same_date,phase1a_003,best_same_date_daytime_only_mlp_w7,0.42,9,314,f1,0.821549,0.813346,0.658328,0.892524
4,same_date,phase1a_003,best_same_date_daytime_only_mlp_w7,0.42,9,314,precision,0.739394,0.732142,0.500000,0.882764
5,same_date,phase1a_003,best_same_date_daytime_only_mlp_w7,0.42,9,314,recall,0.924242,0.928717,0.870350,0.988660
6,same_date,phase1a_002,same_date_daytime_only_gru_w7,0.34,9,314,balanced_accuracy,0.831668,0.834961,0.788572,0.885876
7,same_date,phase1a_002,same_date_daytime_only_gru_w7,0.34,9,314,roc_auc,0.904679,0.909629,0.867073,0.954192
8,same_date,phase1a_002,same_date_daytime_only_gru_w7,0.34,9,314,average_precision,0.838141,0.836085,0.680835,0.944067
9,same_date,phase1a_002,same_date_daytime_only_gru_w7,0.34,9,314,f1,0.810289,0.797139,0.619680,0.887828


saved: c:\workSpace\DeepLearnin_sleep\data\processed\deep_learning_previous_day\experiments\phase_2a_outputs\selected_candidate_participant_bootstrap_summary.csv
saved: c:\workSpace\DeepLearnin_sleep\data\processed\deep_learning_previous_day\experiments\phase_2a_outputs\selected_candidate_participant_bootstrap_details.csv
